# The EM algorithm
EM written out in NumPy (the same code as `images/em_core.py`), checked against *Mathematics for Machine Learning*
Section 11.3 and against scikit-learn; the lower bound; random starts; and the hard-assignment limit.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy import stats
from sklearn.mixture import GaussianMixture

np.set_printoptions(precision=3, suppress=True)
# one seeded random generator for the whole notebook
rng = np.random.default_rng(0)


def weighted_dens(X, pi, mu, cov):
    # pi_k N(x_n | mu_k, cov_k) for every observation n and component k
    return np.column_stack([p * stats.multivariate_normal(m, c).pdf(X) for p, m, c in zip(pi, mu, cov)])


def loglik(X, pi, mu, cov):
    return np.log(weighted_dens(X, pi, mu, cov).sum(axis=1)).sum()


def e_step(X, pi, mu, cov):
    # responsibilities: each row of weighted densities divided by its sum
    W = weighted_dens(X, pi, mu, cov)
    return W / W.sum(axis=1, keepdims=True)


def m_step(X, R):
    Nk = R.sum(axis=0)
    # weighted means first, then covariances around the NEW means, then weights
    mu = (R.T @ X) / Nk[:, None]
    cov = np.array([((R[:, k, None] * (X - mu[k])).T @ (X - mu[k])) / Nk[k] for k in range(R.shape[1])])
    return Nk / len(X), mu, cov

## 1. The seven-observation example

In [ ]:
X7 = np.array([-3, -2.5, -1, 0, 2, 4, 5.0])[:, None]
pi, mu, cov = np.ones(3) / 3, np.array([[-4.0], [0.0], [8.0]]), np.array([[[1.0]], [[0.2]], [[3.0]]])
print("iteration 0: log-likelihood", round(loglik(X7, pi, mu, cov), 3))
for it in range(1, 6):
    R = e_step(X7, pi, mu, cov)
    pi, mu, cov = m_step(X7, R)
    print(f"iteration {it}: weights {pi}, means {mu.ravel()}, variances {cov.ravel()}, "
          f"log-likelihood {loglik(X7, pi, mu, cov):.3f}")

## 2. Two dimensions: Old Faithful
Eruption duration and waiting time (minutes), saved from seaborn's `geyser` dataset.

In [ ]:
import pandas as pd
X = pd.read_csv("data/old_faithful.csv")[["duration", "waiting"]].to_numpy(float)
# a deliberately poor start: centres where there are no eruptions
pi, mu, cov = np.ones(2) / 2, np.array([[2.0, 90.0], [4.5, 50.0]]), np.array([np.diag([1.0, 100.0])] * 2)
lls = [loglik(X, pi, mu, cov)]
for it in range(60):
    pi, mu, cov = m_step(X, e_step(X, pi, mu, cov))
    lls.append(loglik(X, pi, mu, cov))
print("log-likelihood at iterations 0, 1, 5, 10, 60:", np.round([lls[i] for i in (0, 1, 5, 10, 60)], 1))
print("weights", pi, "\nmeans\n", mu)
gm = GaussianMixture(2, n_init=5, random_state=0).fit(X)
print("scikit-learn log-likelihood:", round(gm.score(X) * len(X), 1))
fig = go.Figure(go.Scatter(y=lls, mode="lines+markers"))
fig.update_layout(template="simple_white", xaxis_title="iteration", yaxis_title="log-likelihood", height=350)
fig.show()

## 3. The log-likelihood never decreases, and the lower bound touches it

In [ ]:
steps = np.diff(lls)
print("any change below -1e-9 in 60 iterations?", bool((steps < -1e-9).any()))
# B(theta; q) with q = responsibilities of the same theta, seven observations at the start
pi, mu, cov = np.ones(3) / 3, np.array([[-4.0], [0.0], [8.0]]), np.array([[[1.0]], [[0.2]], [[3.0]]])
W = weighted_dens(X7, pi, mu, cov)
q = W / W.sum(axis=1, keepdims=True)
# 0 log 0 counts as 0
B = np.sum(np.where(q > 0, q * (np.log(W) - np.log(np.where(q > 0, q, 1))), 0))
print("bound B =", round(B, 3), "  log-likelihood =", round(loglik(X7, pi, mu, cov), 3))
# with other weights q the bound is strictly lower
q_flat = np.ones_like(q) / 3
print("bound with q = 1/3 everywhere:", round(np.sum(q_flat * (np.log(W) - np.log(q_flat))), 3))

## 4. Local maxima: 20 random starts
First Old Faithful (every start ends at the same answer), then a constructed dataset with three clusters, two
long and tilted and one round, where some starts get stuck.

In [ ]:
def run_from(X, K, idx, iters=300):
    p, m, c = np.ones(K) / K, X[idx], np.array([np.cov(X.T)] * K)
    for it in range(iters):
        p, m, c = m_step(X, e_step(X, p, m, c))
    return round(loglik(X, p, m, c), 1)


print("Old Faithful:", sorted({run_from(X, 2, rng.choice(len(X), 2, replace=False)) for s in range(20)}))
r5 = np.random.default_rng(5)
covs = [np.array([[6, 2.6], [2.6, 1.4]])] * 2 + [np.eye(2) * 0.6]
means = [np.array([0, 0]), np.array([0, 4]), np.array([6, -2])]
X3 = np.vstack([r5.multivariate_normal(m, c, n) for m, c, n in zip(means, covs, [200, 200, 100])])
finals = []
for s in range(20):
    idx = rng.choice(len(X3), 3, replace=False)
    p, m, c = np.ones(3) / 3, X3[idx], np.array([np.eye(2)] * 3)
    for it in range(200):
        p, m, c = m_step(X3, e_step(X3, p, m, c))
    finals.append(round(loglik(X3, p, m, c), 1))
print("three clusters:", sorted(finals))

## 5. Hard assignment: responsibilities as sigma shrinks
Centres 0 and 4, equal weights, observation at x = 1.

In [ ]:
for s in (2, 1, 0.5, 0.1):
    a, b = stats.norm(0, s).logpdf(1), stats.norm(4, s).logpdf(1)
    print(f"sigma = {s}: responsibility of the centre at 0 = {1 / (1 + np.exp(b - a)):.6f}")